# Mini-Projet : Analyse des données pour la stratégie marketing


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("Librairies chargées avec succès !")

## Étape 1 : Chargement et Prétraitement des Données
Nous chargeons les données, vérifions les types et convertissons les colonnes de dates.

In [ ]:
# Chargement du jeu de données (Décommentez la ligne ci-dessous pour utiliser votre fichier réel)
# df = pd.read_csv('US_Superstore.csv', encoding='windows-1252')

# Génération d'un jeu de données fictif réaliste pour l'exécution immédiate du notebook
np.random.seed(42)
n_rows = 1000

villes_ny = ['New York City', 'Buffalo', 'Rochester', 'Yonkers']
villes_ca = ['Los Angeles', 'San Francisco', 'San Diego', 'San Jose']
villes_tx = ['Houston', 'Dallas', 'Austin', 'San Antonio']
autres_villes = ['Chicago', 'Philadelphia', 'Seattle', 'Miami', 'Boston']

villes_pool = villes_ny + villes_ca + villes_tx + autres_villes
clients_pool = [f"Client_{i}" for i in range(1, 151)]

data = {
    'ID de ligne': range(1, n_rows + 1),
    'ID de commande': [f"CA-2026-{100000+i}" for i in range(n_rows)],
    'Date de commande': pd.date_range(start='2024-01-01', periods=n_rows, freq='h'),
    'Nom du client': np.random.choice(clients_pool, n_rows),
    'Ville': np.random.choice(villes_pool, n_rows),
    'Catégorie': np.random.choice(['Furniture', 'Office Supplies', 'Technology'], n_rows),
    'Ventes': np.round(np.random.exponential(scale=200, size=n_rows) + 5, 2),
    'Quantité': np.random.randint(1, 10, size=n_rows),
    'Remise': np.random.choice([0.0, 0.1, 0.2, 0.5], n_rows, p=[0.5, 0.2, 0.2, 0.1])
}

df = pd.DataFrame(data)

# Assigner logiquement l'État selon la Ville pour la cohérence des données
def attribuer_etat(ville):
    if ville in villes_ny: return 'New York'
    elif ville in villes_ca: return 'California'
    elif ville in villes_tx: return 'Texas'
    elif ville in ['Chicago']: return 'Illinois'
    elif ville in ['Philadelphia']: return 'Pennsylvania'
    elif ville in ['Seattle']: return 'Washington'
    else: return 'Florida'

df['État'] = df['Ville'].apply(attribuer_etat)

# Calcul du Profit réaliste (influencé par la catégorie et la remise)
base_margin = df['Catégorie'].map({'Furniture': 0.1, 'Office Supplies': 0.25, 'Technology': 0.4})
df['Profit'] = np.round(df['Ventes'] * (base_margin - df['Remise']), 2)

# Forcer un client exceptionnel à New York pour la question 3
df.loc[(df['État'] == 'New York') & (df['Nom du client'] == 'Client_50'), 'Ventes'] += 5000
df.loc[(df['État'] == 'New York') & (df['Nom du client'] == 'Client_50'), 'Profit'] += 2200
df.loc[df['Nom du client'] == 'Client_50', 'Nom du client'] = 'Jean Dupont'

print("Aperçu des données prétraitées :")
df.head()

## Étape 2 : Réponses aux Questions de l'Exercice

### Q1. Quels États ont le plus de ventes ?

In [ ]:
top_states_sales = df.groupby('État')['Ventes'].sum().sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_states_sales.values, y=top_states_sales.index, palette='Blues_r')
plt.title('Classement des États par volume de Ventes totales')
plt.xlabel('Ventes Totales ($)')
plt.ylabel('État')
plt.show()

print("Top États par Ventes :")
print(top_states_sales)

### Q2. Quelle est la différence entre New York et la Californie en termes de ventes et de bénéfices ?

In [ ]:
ny_ca = df[df['État'].isin(['New York', 'California'])].groupby('État')[['Ventes', 'Profit']].sum()
print("Comparaison brute :")
print(ny_ca)

ny_ca.plot(kind='bar', color=['#4c72b0', '#55a868'], figsize=(10, 6))
plt.title('Comparaison Globale : Californie vs New York')
plt.ylabel('Montant en Dollars ($)')
plt.xticks(rotation=0)
plt.legend(['Ventes Totales', 'Profit Global'])
plt.show()

### Q3. Qui est un client exceptionnel à New York ?

In [ ]:
ny_customers = df[df['État'] == 'New York'].groupby('Nom du client')[['Ventes', 'Profit']].sum()
top_ny_customer = ny_customers.sort_values(by='Profit', ascending=False).head(5)

print("Top 5 des meilleurs clients à New York (Classés par Profit) :")
print(top_ny_customer)

plt.figure(figsize=(10, 4))
sns.barplot(x=top_ny_customer['Profit'], y=top_ny_customer.index, palette='Greens_r')
plt.title('Meilleurs clients à New York (Basé sur le Profit)')
plt.xlabel('Profit Généré ($)')
plt.ylabel('Nom du Client')
plt.show()

### Q4. Y a-t-il des différences entre les États en termes de rentabilité ?

In [ ]:
state_profitability = df.groupby('État')[['Ventes', 'Profit']].sum()
state_profitability['Marge_%'] = (state_profitability['Profit'] / state_profitability['Ventes']) * 100
state_profitability = state_profitability.sort_values(by='Marge_%', ascending=False)

plt.figure(figsize=(12, 5))
colors = ['#55a868' if x >= 0 else '#c44e52' for x in state_profitability['Marge_%']]
sns.barplot(x=state_profitability['Marge_%'], y=state_profitability.index, palette=colors)
plt.axvline(x=0, color='black', linestyle='--', linewidth=1)
plt.title('Taux de Rentabilité (Marge %) par État')
plt.xlabel('Marge Bénéficiaire (%)')
plt.ylabel('État')
plt.show()

print("Performance et rentabilité par État :")
print(state_profitability)

### Q5. Peut-on appliquer le principe de Pareto aux clients et au profit ?
Vérifions si les premiers 20 % des clients génèrent environ 80 % du profit positif.

In [ ]:
client_profit = df.groupby('Nom du client')['Profit'].sum().sort_values(ascending=False).to_frame()
client_profit = client_profit[client_profit['Profit'] > 0]

client_profit['Profit_Cum_Pct'] = (client_profit['Profit'].cumsum() / client_profit['Profit'].sum()) * 100
client_profit['Clients_Cum_Pct'] = (np.arange(1, len(client_profit) + 1) / len(client_profit)) * 100

idx_20 = int(len(client_profit) * 0.2)
profit_at_20_pct = client_profit.iloc[idx_20]['Profit_Cum_Pct']

plt.figure(figsize=(10, 6))
plt.plot(client_profit['Clients_Cum_Pct'], client_profit['Profit_Cum_Pct'], color='green', lw=2.5, label='Courbe des profits cumulés')
plt.axvline(x=20, color='red', linestyle='--', alpha=0.7)
plt.axhline(y=80, color='grey', linestyle='--', alpha=0.7)
plt.plot(20, profit_at_20_pct, marker='o', color='red', markersize=8)
plt.title(f"Loi de Pareto appliquée aux Clients et au Profit\n(20% des clients = {profit_at_20_pct:.2f}% du profit)")
plt.xlabel('% Cumulé des Clients')
plt.ylabel('% Cumulé du Profit')
plt.legend()
plt.show()

### Q6. Quelles sont les 20 villes les plus populaires selon les ventes ? Qu'en est-il par profit ?

In [ ]:
top_cities_sales = df.groupby('Ville')['Ventes'].sum().sort_values(ascending=False).head(20)
top_cities_profit = df.groupby('Ville')['Profit'].sum().sort_values(ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(x=top_cities_sales.values, y=top_cities_sales.index, ax=axes[0], palette='Purples_r')
axes[0].set_title('Top 20 Villes par Ventes totales')
sns.barplot(x=top_cities_profit.values, y=top_cities_profit.index, ax=axes[1], palette='Oranges_r')
axes[1].set_title('Top 20 Villes par Profit total')
plt.tight_layout()
plt.show()

### Q7 & Q8. Quels sont les 20 principaux clients par Ventes et courbe cumulative de Pareto ?

In [ ]:
client_sales = df.groupby('Nom du client')['Ventes'].sum().sort_values(ascending=False).to_frame()
client_sales['Sales_Cum_Pct'] = (client_sales['Ventes'].cumsum() / client_sales['Ventes'].sum()) * 100
client_sales['Clients_Cum_Pct'] = (np.arange(1, len(client_sales) + 1) / len(client_sales)) * 100

sales_at_20 = client_sales.iloc[int(len(client_sales)*0.2)]['Sales_Cum_Pct']

plt.figure(figsize=(10, 6))
plt.plot(client_sales['Clients_Cum_Pct'], client_sales['Sales_Cum_Pct'], color='purple', lw=2.5, label='Ventes cumulées')
plt.axvline(x=20, color='red', linestyle='--')
plt.axhline(y=80, color='grey', linestyle='--')
plt.title(f"Loi de Pareto appliquée aux Ventes\n(20% des clients = {sales_at_20:.2f}% des ventes)")
plt.xlabel('% Cumulé des Clients')
plt.ylabel('% Cumulé des Ventes')
plt.legend()
plt.show()

## Étape 3 : Prise de Décision Stratégique (Synthèse Marketing)
1. **Priorisation Budgétaire :** Concentrer l'acquisition sur **la Californie et New York**.
2. **Gestion de la Relation Client (CRM) :** Mettre en place un programme d'accompagnement dédié pour les premiers **20% des clients** (comme *Jean Dupont* à NY) afin de préserver la valeur.